## Preprocessing

- Added time features (hour, day, weekend flag)
- Extracted caps, emoji, punctuation signals from raw text
- Cleaned text separately for TF-IDF
- Added sentiment labels using VADER (negative / non-negative)
- Capped response time at 500 mins, applied log transform
- Final: 1.25M rows, 25 columns

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

# ML
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from sklearn.pipeline import Pipeline

# NLP
import nltk
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords


## load data

In [2]:
merged = pd.read_csv("../data/raw/merged_with_text.csv")
print(f"Base shape: {merged.shape}")
print(merged.columns.tolist())

Base shape: (1261885, 10)
['tweet_id_customer', 'created_at_customer', 'author_id', 'tweet_id_company', 'created_at_company', 'in_response_to_tweet_id', 'response_time_mins', 'year_responded', 'customer_text', 'company_text']


## pre processing

In [3]:
merged["created_at_customer"] = pd.to_datetime(merged["created_at_customer"])
merged["hour"] = merged["created_at_customer"].dt.hour
merged["day_of_week"] = merged["created_at_customer"].dt.day_name()
merged["is_weekend"] = merged["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

print(merged[["created_at_customer", "hour", "day_of_week", "is_weekend"]].head(3))

        created_at_customer  hour day_of_week  is_weekend
0 2017-10-31 22:08:27+00:00    22     Tuesday           0
1 2017-10-31 21:49:35+00:00    21     Tuesday           0
2 2017-10-31 21:45:10+00:00    21     Tuesday           0


In [4]:
# clean text 
def extract_raw_features(text):
    text = str(text)
    letters = [c for c in text if c.isalpha()]
    caps_ratio = sum(1 for c in letters if c.isupper()) / max(len(letters), 1)
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F9FF"
        u"\U00002700-\U000027BF"
        "]+", flags=re.UNICODE)
    return {
        "caps_ratio": round(caps_ratio, 3),
        "has_emoji": int(bool(emoji_pattern.search(text))),
        "has_exclamation": int("!" in text),
        "has_question": int("?" in text),
        "exclamation_count": text.count("!"),
        "question_count": text.count("?"),
        "text_length": len(text)
    }

def clean_text(text):
    text = re.sub(r'@\w+', '', str(text))
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^\w\s!?]', '', text)
    text = text.lower().strip()
    return text

print("Extracting features...")
raw_features = merged["customer_text"].apply(extract_raw_features)
feature_df = pd.DataFrame(raw_features.tolist(), index=merged.index)
merged = pd.concat([merged, feature_df], axis=1)

merged["customer_text_clean"] = merged["customer_text"].apply(clean_text)
merged["customer_text_clean"] = merged["customer_text_clean"].fillna("").str.strip()


merged["company_text_clean"] = merged["company_text"].apply(clean_text)
merged["company_text_clean"] = merged["company_text_clean"].fillna("").str.strip()

print(f"Shape: {merged.shape}")


Extracting features...
Shape: (1261885, 22)


In [5]:
print(merged[["customer_text", "customer_text_clean", "company_text", "company_text_clean"]].head(3))


                                       customer_text  \
0  @sprintcare I have sent several private messag...   
1                                 @sprintcare I did.   
2          @sprintcare is the worst customer service   

                                 customer_text_clean  \
0  i have sent several private messages and no on...   
1                                              i did   
2                      is the worst customer service   

                                        company_text  \
0  @115712 I understand. I would like to assist y...   
1  @115712 Please send us a Private Message so th...   
2  @115712 Can you please send us a private messa...   

                                  company_text_clean  
0  i understand i would like to assist you we wou...  
1  please send us a private message so that we ca...  
2  can you please send us a private message so th...  


In [7]:
#add sentiment labels using 
sia = SentimentIntensityAnalyzer() 

def get_sentiment(text):
    score = sia.polarity_scores(str(text))["compound"]
    if score <= -0.3:
        return "negative"
    else:
        return "non-negative"

merged["sentiment"] = merged["customer_text"].apply(get_sentiment)
print(merged["sentiment"].value_counts())
print(f"\n{merged['sentiment'].value_counts(normalize=True).round(3)}")

sentiment
non-negative    969093
negative        292792
Name: count, dtype: int64

sentiment
non-negative    0.768
negative        0.232
Name: proportion, dtype: float64


In [8]:
# regression label (remove the outliers)
merged["response_time_capped"] = merged["response_time_mins"].clip(upper=500)
merged["response_time_log"] = np.log1p(merged["response_time_capped"])

print(merged[["response_time_mins", "response_time_capped", "response_time_log"]].describe())

       response_time_mins  response_time_capped  response_time_log
count        1.261885e+06          1.261885e+06       1.261885e+06
mean         2.936730e+02          1.028362e+02       3.361872e+00
std          5.006159e+03          1.617182e+02       1.671608e+00
min          1.666667e-02          1.666667e-02       1.652930e-02
25%          6.200000e+00          6.200000e+00       1.974081e+00
50%          2.118333e+01          2.118333e+01       3.099341e+00
75%          1.073000e+02          1.073000e+02       4.684905e+00
max          2.609747e+06          5.000000e+02       6.216606e+00


In [14]:
# add company name
df_original = pd.read_csv("../data/twcs/twcs.csv")
company_lookup = df_original[df_original["inbound"]==False][["tweet_id", "author_id"]].rename(
    columns={"author_id": "company_name"}
)
merged = merged.merge(
    company_lookup,
    left_on="tweet_id_company",
    right_on="tweet_id",
    how="left"
).drop("tweet_id", axis=1)

print(merged["company_name"].value_counts().head(5))

company_name
AmazonHelp      167310
AppleSupport    105888
Uber_Support     55705
SpotifyCares     42573
Delta            41873
Name: count, dtype: int64


In [15]:
#save 
merged = merged.dropna(subset=["customer_text_clean"])
merged = merged[merged["customer_text_clean"] != ""]

os.makedirs("../data/processed", exist_ok=True)
merged.to_csv("../data/processed/full_data.csv", index=False)

print(f"Final shape: {merged.shape}")
print(f"Columns: {merged.columns.tolist()}")


Final shape: (1252018, 26)
Columns: ['tweet_id_customer', 'created_at_customer', 'author_id', 'tweet_id_company', 'created_at_company', 'in_response_to_tweet_id', 'response_time_mins', 'year_responded', 'customer_text', 'company_text', 'hour', 'day_of_week', 'is_weekend', 'caps_ratio', 'has_emoji', 'has_exclamation', 'has_question', 'exclamation_count', 'question_count', 'text_length', 'customer_text_clean', 'company_text_clean', 'sentiment', 'response_time_capped', 'response_time_log', 'company_name']


In [16]:
from sklearn.model_selection import train_test_split
import os


train_df, temp_df = train_test_split(
    merged, test_size=0.3, random_state=42,
    stratify=merged["sentiment"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42,
    stratify=temp_df["sentiment"]
)

train_df.to_csv("../data/processed/train.csv", index=False)
val_df.to_csv("../data/processed/val.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

print(f"Train: {train_df.shape}")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")
print(f"\nSentiment in train:")
print(train_df["sentiment"].value_counts(normalize=True).round(3))

Train: (876412, 26)
Val:   (187803, 26)
Test:  (187803, 26)

Sentiment in train:
sentiment
non-negative    0.766
negative        0.234
Name: proportion, dtype: float64
